In [1]:
using LinearAlgebra
using LinearOperators
using SparseArrays

In [2]:
function get_1d_laplace_op_matrix(n)
	off = ones(n-1)
	diag = ones(n)
	spdiagm(-1 => off, 0 => -2diag, 1 => off)
end

get_1d_laplace_op_matrix (generic function with 1 method)

In [3]:
# L
# E L, L E
# E E L, E L E, L E E
# E E E L, E E L E, E L E E, L E E E
# (E E E L), (E E L) (E), (E L) (E E), (L) (E E E)
# E ... E L = L of size n^(k+1) if E is there k times
function get_laplace_op_matrix(n, d)
	L_full = get_1d_laplace_op_matrix(n^d)
	for k in 1:(d-1)
		L = get_1d_laplace_op_matrix(n^(d-k))
		size = n^k
		E = sparse(I, size, size)
		L_full += kron(L, E)
	end
	L_full
end

# 4 points, 3 dims
#get_laplace_op_matrix(4, 3)

get_laplace_op_matrix (generic function with 1 method)

In [4]:
#Matrix(L_full)

## define RHS and solve using different methods

$\Delta u = f \:\:\text{on}\:\:\Omega$

$u = g \:\:\text{on}\:\:\partial\Omega$

supp 1D:

$ \partial u / \partial x \approx \frac{u(x+h/2) - u(x-h/2)}{h}$

$\Delta u(x) \approx \frac{u(x+h) - 2u(x) + u(x-h)}{h^2}$

$ u_{i-1} - 2u_i + u_{i+1} = h^2 f_i$

$ i = 0,1,...,n,n+1 $

boundary: $\:i=0, \:i=n+1 $

we know $\:u_0, \:u_{n+1}\:$ from BC

we need $n$ equation for $n$ unknowns ($u_1,...,u_n$)

$ u_0 - 2u_1 + u_2 = h^2 f_1 $

$ u_1 - 2u_2 + u_3 = h^2 f_2 $

...

$ u_{n-2} - 2u_{n-1} + u_{n} = h^2 f_{n-1} $

$ u_{n-1} - 2u_n + u_{n+1} = h^2 f_n $

we get

$ L U = F $

where
- $L$ matrix of size $n \times n$
- $U = (u_1, u_2, ..., u_{n-1}, u_n)$
- $F = h^2(f_1, f_2, ..., f_{n-1}, f_n) - (u_0, 0, ..., 0, u_{n+1})$

In [5]:
n = 100
d = 2
N = n^d

10000

In [6]:
h = 1 / (n+1)

0.009900990099009901

In [7]:
function u_analytic_fun(x)
    #prod(x)*prod(1 .- x)
    prod(sin.(pi*x))
end

function f_fun(x)
    d = length(x)
    -d*pi^2 * prod(sin.(pi*x))
end

f_fun (generic function with 1 method)

### fiddling around

In [8]:
v = Vector([1,2,3])
v, f_fun(v)

([1, 2, 3], 3.262907826717048e-46)

In [9]:
v = Vector([Vector([1,2,3]), Vector([1,1,1])])
v, f_fun.(v)

([[1, 2, 3], [1, 1, 1]], [3.262907826717048e-46, -5.438179711195081e-47])

In [10]:
xs = [range(h, 1-h; length=n) for _ in 1:d]

2-element Vector{StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}}:
 0.009900990099009901:0.009900990099009901:0.9900990099009901
 0.009900990099009901:0.009900990099009901:0.9900990099009901

In [11]:
coords = collect(Iterators.product(xs...))

100×100 Matrix{Tuple{Float64, Float64}}:
 (0.00990099, 0.00990099)  (0.00990099, 0.019802)  …  (0.00990099, 0.990099)
 (0.019802, 0.00990099)    (0.019802, 0.019802)       (0.019802, 0.990099)
 (0.029703, 0.00990099)    (0.029703, 0.019802)       (0.029703, 0.990099)
 (0.039604, 0.00990099)    (0.039604, 0.019802)       (0.039604, 0.990099)
 (0.049505, 0.00990099)    (0.049505, 0.019802)       (0.049505, 0.990099)
 (0.0594059, 0.00990099)   (0.0594059, 0.019802)   …  (0.0594059, 0.990099)
 (0.0693069, 0.00990099)   (0.0693069, 0.019802)      (0.0693069, 0.990099)
 (0.0792079, 0.00990099)   (0.0792079, 0.019802)      (0.0792079, 0.990099)
 (0.0891089, 0.00990099)   (0.0891089, 0.019802)      (0.0891089, 0.990099)
 (0.0990099, 0.00990099)   (0.0990099, 0.019802)      (0.0990099, 0.990099)
 (0.108911, 0.00990099)    (0.108911, 0.019802)    …  (0.108911, 0.990099)
 (0.118812, 0.00990099)    (0.118812, 0.019802)       (0.118812, 0.990099)
 (0.128713, 0.00990099)    (0.128713, 0.019802)     

In [12]:
collect(coords[1])

2-element Vector{Float64}:
 0.009900990099009901
 0.009900990099009901

In [13]:
[collect(coords[1]), collect(coords[2])]

2-element Vector{Vector{Float64}}:
 [0.009900990099009901, 0.009900990099009901]
 [0.019801980198019802, 0.009900990099009901]

In [14]:
f_fun.([collect(coords[1]), collect(coords[2])])

2-element Vector{Float64}:
 -0.019091791043757085
 -0.03816511201270312

In [15]:
#grid_points_as_1d_vect = [collect(x) for x in coords[1:end]]
#f_fun.(grid_points_as_1d_vect)

### here we go

In [16]:
function get_grid_points_as_1d_vect(n, d)
    a = 0
    b = 1
    h = 1/(n+1)
    xs = [range(h, 1-h; length=n) for _ in 1:d]
    coords = collect(Iterators.product(xs...))
    [collect(x) for x in coords[1:end]]
end
#f.(grid_points_as_1d_vect)

get_grid_points_as_1d_vect (generic function with 1 method)

In [17]:
n^d

10000

In [18]:
grid_points_as_1d_vect = get_grid_points_as_1d_vect(n,d);

In [19]:
U_analytic = u_analytic_fun.(grid_points_as_1d_vect);

In [20]:
f = f_fun.(grid_points_as_1d_vect);
F = h^2 * f;

In [21]:
L = get_laplace_op_matrix(n,d);

In [22]:
using IterativeSolvers
U_cg = cg(L, F)

10000-element Vector{Float64}:
 0.01928132630789413
 0.02633233541903
 0.029419758808380887
 0.031039314542434384
 0.03203835238350517
 0.03274289792281522
 0.0332944641374758
 0.03376132516090135
 0.034179370023113255
 0.03456873581498859
 0.03494125564067625
 0.03530408418547458
 0.03566159320233715
 ⋮
 0.0353040841854746
 0.03494125564067628
 0.0345687358149886
 0.03417937002311324
 0.0337613251609013
 0.03329446413747585
 0.03274289792281526
 0.03203835238350517
 0.03103931454243434
 0.02941975880838084
 0.026332335419029986
 0.019281326307894094

In [23]:
U_direct = L \ F;

In [24]:
using CairoMakie
CairoMakie.activate!()

In [25]:
#grid_points = collect(range(h,1-h, length=n));
#scatter(grid_points, U_analytic, color=:gray0)
#lines!(grid_points, U_direct, color=:royalblue1)
#lines!(grid_points, U_cg, color=:gray0)
#current_figure()

In [26]:
#n_trials = 8
#n_mult = 8
#max_jacobi = zeros(n_trials)
#max_gs = zeros(n_trials)
#max_direct = zeros(n_trials)
#
#for i in 1:n_trials
#    n = n_mult*i
#    h = 1/(n+1)
#    grid_points = collect(range(h,1-h, length=n))
#    L = get_laplace_op_matrix(n)
#    F = h^2 * f_fun(grid_points)
#    U_jacobi = jacobi(L, F)
#    U_gs = gauss_seidel(L, F; maxiter=100)
#    U_direct = L \ F
#
#    max_jacobi[i] = maximum(U_jacobi)
#    max_gs[i] = maximum(U_gs)
#    max_direct[i] = maximum(U_direct)
#end